# Import libraries

In [1]:
import pyvisa
import numpy as np
import json
import time
import struct
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.ndimage import gaussian_filter

# Functions

In [2]:
#Returns the voltage scaling in SI unit from the string read from the scope.

def parse_volt_per_div(s):
    """
    Owob scale string → volt/div (float)
    pl: '500mV->', '2V->', '200uV->'
    """
    s = s.replace('->', '').strip()

    if s.endswith('mV'):
        return float(s[:-2]) * 1e-3
    elif s.endswith('uV'):
        return float(s[:-2]) * 1e-6
    elif s.endswith('V'):
        return float(s[:-1])
    else:
        raise ValueError(f"Unknown voltage format: {s}")

#====================================================================================================
#====================================================================================================
#====================================================================================================

#Converts the datapoints into voltage values.

def adc_to_volt(adc, v_div, adc_bits=12):
    full_scale = v_div * 10 #10 total divs in voltage axis.
    lsb = full_scale / (2**adc_bits)
    
    adc_mid = np.mean(adc) #Removing offset.
    return (adc - adc_mid) * lsb

#====================================================================================================
#====================================================================================================
#====================================================================================================

#Returns the time scaling in SI unit from the string read from the scope.

def parse_time_per_div(s):
    s = s.replace('->', '').strip()

    if s.endswith('ns'):
        return float(s[:-2]) * 1e-9
    elif s.endswith('us'):
        return float(s[:-2]) * 1e-6
    elif s.endswith('ms'):
        return float(s[:-2]) * 1e-3
    elif s.endswith('s'):
        return float(s[:-1])
    else:
        raise ValueError(f"Unknown time format: {s}")

#====================================================================================================
#====================================================================================================
#====================================================================================================
        
#This bruteforce code stops the scope, reads the deepmemory data from CH1 and CH2 input. 
#The raw data is transformed the correct voltage and time values, which also returned and saved into a file.

#-----Inputs.-----
#scope: initialized scope name (rm.open_resource type, e.g.: "scope")
#ip_address: string of the scope IP address (e.g.: "TCPIP0::192.168.1.72::3000::SOCKET")
#num_of_cycles: The number of curves read from the scope.
#wait_time: The time in s to be waited between restarting (running) and stopping the scope.
#filename: string of the desired filename to save the data in .txt format (e.g.: "scope_data.txt").

#-----Outputs.-----
#scope: initialized scope name (rm.open_resource type, e.g.: "scope")
#truncated_list: 3D pyhton list, where 1st index is the one of the cycle, 2nd one is the coloumn (0=time, 1=ch1 voltage, 2=ch2 voltage), 3rd one is the datapoint

def read_deepmemory_data(scope, ip_address, num_of_cycles, wait_time, filename):
    results = []
    
    chunk_size = 40000
    timeout = 1000
    
    for cycle in np.arange(0,num_of_cycles):
        scope.write(":TRIGger:FORCe")
        time.sleep(0.1)
        scope.write(":RUNning STOP")
        time.sleep(0.1)
        scope.write(":ACQuire:DEPMEM 10K")
    
        #Read the scaling of the input channels and the time axis.
        scope.clear()
        ch1_scale_str = scope.query(":CH1:SCAle?")
        time.sleep(0.1)
        scope.clear()
        ch2_scale_str = scope.query(":CH2:SCAle?")
        time.sleep(0.1)
        scope.clear()
        time_scale_str = scope.query(":HORIzontal:SCAle?")
        time.sleep(0.1)
        
        scope.timeout = timeout #3 seconds timeout to read from deep memory.
        scope.chunk_size = chunk_size # 3.6 kByte buffer.
        
        #-----Read CH1 input.-----
        scope.write(":DATA:WAVE:DEPMem:CH1?")
        
        #Wait.
        time.sleep(0.1)
        
        #Read the raw data in chunks.
        raw_all_1 = b""
        while True:
            try:
                chunk_1 = scope.read_raw()
                raw_all_1 += chunk_1  
                #If the data is less than the buffer, we stop.
                if len(chunk_1) == 0:
                    break
            except pyvisa.errors.VisaIOError:
                #If there is timeout, there is no more data.
                break
        
        scope.close() #Stops the scope.
        
        #Reconnect the scope.
        rm = pyvisa.ResourceManager()
        scope = rm.open_resource(ip_address)
        
        scope.timeout = timeout
        scope.chunk_size = chunk_size
        
        scope.write_termination = '\r\n'
        scope.query_termination = '\r\n'
        scope.read_termination = '\r\n'
        
        scope.clear()
        scope.write("*CLS") #Clear the register.
        
        time.sleep(0.1)
    
        #-----Read CH2 input.-----
        scope.write(":DATA:WAVE:DEPMem:CH2?")
        
        #Wait.
        time.sleep(0.1)
        
        #Read the raw data in chunks.
        raw_all_2 = b""
        while True:
            try:
                chunk_2 = scope.read_raw()
                raw_all_2 += chunk_2  
                #If the data is less than the buffer, we stop.
                if len(chunk_2) == 0:
                    break
            except pyvisa.errors.VisaIOError:
                #If there is timeout, there is no more data.
                break        
        
        print(f"Total read bytes on CH1: {len(raw_all_1)}")
        print(f"Total read bytes on CH2: {len(raw_all_2)}")
        print(f"Cycle: {cycle}")
        
        scope.close() #Stops the scope.
    
        #-----Restart the scope, enable run mode.-----
    
        rm = pyvisa.ResourceManager()
        scope = rm.open_resource(ip_address)
        
        scope.timeout = timeout
        scope.chunk_size = chunk_size
        
        scope.write_termination = '\r\n'
        scope.query_termination = '\r\n'
        scope.read_termination = '\r\n'
        
        scope.clear()
        scope.write("*CLS") #Clear the register. 
        
        scope.write(":RUNning RUN") #Run the scope.
        
        time.sleep(0.1)
    
        #-----Data procession.-----
    
        #Cut the headers.
        cropped_raw_1 = raw_all_1[4:] 
        cropped_raw_2 = raw_all_2[4:] 
        
        #Calculate the number of datapoints.
        num_elements_1 = len(cropped_raw_1) // 2
        num_elements_2 = len(cropped_raw_2) // 2
    
        #Unpack the data by "struct.unpack".
        if num_elements_1 > 0:
            ch1_data = struct.unpack('<' + ('h' * num_elements_1), cropped_raw_1[:num_elements_1*2])
        else:
            print("Not enough data for CH1, try with higher voltage window.")
            
            scope.close() #Stops the scope.
    
            #-----Restart the scope, enable run mode.-----
    
            rm = pyvisa.ResourceManager()
            scope = rm.open_resource(ip_address)
        
            scope.timeout = 5000
        
            scope.write_termination = '\r\n'
            scope.query_termination = '\r\n'
            scope.read_termination = '\r\n'
        
            scope.clear()
            scope.write("*CLS") #Clear the register. 
        
            scope.write(":RUNning RUN") #Run the scope.
        
            time.sleep(0.1)
            empty=[]
            return scope, empty
        
        if num_elements_2 > 0:
            ch2_data = struct.unpack('<' + ('h' * num_elements_2), cropped_raw_2[:num_elements_2*2])
        else:
            print("Not enough data for CH2, try with higher voltage window.")
            scope.close() #Stops the scope.
    
            #-----Restart the scope, enable run mode.-----
    
            rm = pyvisa.ResourceManager()
            scope = rm.open_resource(ip_address)
        
            scope.timeout = timeout
        
            scope.write_termination = '\r\n'
            scope.query_termination = '\r\n'
            scope.read_termination = '\r\n'
        
            scope.clear()
            scope.write("*CLS") #Clear the register. 
        
            scope.write(":RUNning RUN") #Run the scope.
        
            time.sleep(0.1)
            return scope, empty
    
        #Check which list is longer, CH1 input or CH2 input data.
        min_length=np.min([len(ch1_data), len(ch2_data)])
        max_length=np.max([len(ch1_data), len(ch2_data)])
    
        #Recover voltage from raw data.
        ch1_volt = adc_to_volt(ch1_data, parse_volt_per_div(ch1_scale_str))[0:min_length]
        ch2_volt = adc_to_volt(ch2_data, parse_volt_per_div(ch2_scale_str))[0:min_length]
        
        #Recover time from raw data.
        sample_rate = min_length / (parse_time_per_div(time_scale_str) * (20)*(min_length/max_length)) #20 total divs in time axis.
        t = np.arange(min_length) / sample_rate #The time scaling is compensated with the unequal read values from CH1 and CH2 inputs.
    
        #Stack the data to n x 3 shape.
        #data_out = np.column_stack((t, ch1_volt, ch2_volt))
        
        #Save to file.
        # np.savetxt(
            # filename+"_"+str(cycle)+".txt",
            # data_out,
            # delimiter="\t",
            # header="time[s]\tCH1[V]\tCH2[V]",
            # comments=''
        # )
        results.append([t, ch1_volt, ch2_volt])
        time.sleep(wait_time)
        
    idx, shortest_list = min(enumerate(results[i][0] for i in range(len(results))), key=lambda x: len(x[1]))

    truncated_list = [
    [inner[:len(shortest_list)] for inner in middle] 
    for middle in results
    ]
    
    np_results = np.array(truncated_list)
    final_data = np_results.reshape(-1, 3)

    np.savetxt(
        filename,
        final_data,
        delimiter="\t",
        header="time[s]\tCH1[V]\tCH2[V] (num. of cycles: "+str(num_of_cycles)+")",
        comments=""
     )
    
    return scope, truncated_list

#====================================================================================================
#====================================================================================================
#====================================================================================================

#This bruteforce code stops the scope, reads the screen data from CH1 and CH2 input. 
#The raw data is transformed the correct voltage and time values, which also returned and saved into a file.

#-----Inputs.-----
#scope: initialized scope name (rm.open_resource type, e.g.: "scope")
#ip_address: string of the scope IP address (e.g.: "TCPIP0::192.168.1.72::3000::SOCKET")
#num_of_cycles: The number of curves read from the scope.
#wait_time: The time in s to be waited between restarting (running) and stopping the scope.
#filename: string of the desired filename to save the data in .txt format (e.g.: "scope_data.txt").

#-----Outputs.-----
#scope: initialized scope name (rm.open_resource type, e.g.: "scope")
#truncated_list: 3D pyhton list, where 1st index is the one of the cycle, 2nd one is the coloumn (0=time, 1=ch1 voltage, 2=ch2 voltage), 3rd one is the datapoint

def read_screen_data(scope, ip_address, num_of_cycles, wait_time, filename):
    results = []
    
    for i in np.arange(num_of_cycles):
        
        #Define timeout and chunk size (the length of the returned block in bytes).
        timeout=1000
        chunk_size=4+1520*2
        
        scope.write_termination = '\n'
        scope.read_termination = '\n'
        
        scope.write(":TRIGger:FORCe")
        time.sleep(0.1)
        scope.write(":RUNning STOP")
        time.sleep(0.1)
        
        scope.write(':DATA:WAVE:SCREen:HEAD?')
                
        #Wait.
        time.sleep(0.1)
                
        #Read the raw data in chunks.
        raw_all_noob = b""
        while True:
            try:
                chunk = scope.read_bytes(4+1520*2)
                raw_all_noob += chunk  
                #If the data is less than the buffer, we stop.
                if len(chunk) == 0:
                    del locals()[chunk]
                    break
            except pyvisa.errors.VisaIOError:
                        #If there is timeout, there is no more data.
                break
                
        scope.close() #Stops the scope.
        
        #Reconnect the scope.
        rm = pyvisa.ResourceManager()
        scope = rm.open_resource(ip_address)
                
        scope.timeout = timeout
        scope.chunk_size = chunk_size
                
        scope.write_termination = '\n'
        scope.query_termination = '\n'
        scope.read_termination = '\n'
                
        scope.clear()
        scope.write("*CLS") #Clear the register.
                
        
        scope.write(':DATA:WAVE:SCREen:CH1?')
                
        #Wait.
        time.sleep(0.1)
                
        #Read the raw data in chunks.
        raw_all_1 = b""
        while True:
            try:
                chunk = scope.read_bytes(4+1520*2)
                raw_all_1 += chunk  
                #If the data is less than the buffer, we stop.
                if len(chunk) == 0:
                    del locals()[chunk]
                    break
            except pyvisa.errors.VisaIOError:
                        #If there is timeout, there is no more data.
                break
                
        scope.close() #Stops the scope.
        
        #Reconnect the scope.
        rm = pyvisa.ResourceManager()
        scope = rm.open_resource(ip_address)
                
        scope.timeout = timeout
        scope.chunk_size = chunk_size
                
        scope.write_termination = '\r\n'
        scope.query_termination = '\r\n'
        scope.read_termination = '\r\n'
                
        scope.clear()
        scope.write("*CLS") #Clear the register.
        
        time.sleep(0.1)
        
        scope.write(':DATA:WAVE:SCREen:CH2?')
                
        #Wait.
        time.sleep(0.1)
                
        #Read the raw data in chunks.
        raw_all_2 = b""
        while True:
            try:
                chunk = scope.read_bytes(4+1520*2)
                raw_all_2 += chunk  
                #If the data is less than the buffer, we stop.
                if len(chunk) == 0:
                    del locals()[chunk]
                    break
            except pyvisa.errors.VisaIOError:
                        #If there is timeout, there is no more data.
                break
                
        scope.close() #Stops the scope.
        
        #Reconnect the scope.
        rm = pyvisa.ResourceManager()
        scope = rm.open_resource(ip_address)
                
        scope.timeout = timeout
        scope.chunk_size = chunk_size
                
        scope.write_termination = '\r\n'
        scope.query_termination = '\r\n'
        scope.read_termination = '\r\n'
                
        scope.clear()
        scope.write("*CLS") #Clear the register.

        #Get the screen voltage/time division info.
        ch1_scale_str = scope.query(":CH1:SCAle?")
        time.sleep(0.1)
        scope.clear()
        ch2_scale_str = scope.query(":CH2:SCAle?")
        time.sleep(0.1)
        scope.clear()
        time_scale_str = scope.query(":HORIzontal:SCAle?")
        time.sleep(0.1)
        
        scope.write(":RUNning RUN")
                
        time.sleep(0.1)
        
        print(f"Total read bytes on CH1: {len(raw_all_1)}")
        print(f"Total read bytes on CH1: {len(raw_all_2)}")
        print(f"Cycle: {i}")
        
        cropped_raw_1 = raw_all_1[4:] 
        cropped_raw_2 = raw_all_2[4:]
        num_elements_1 = len(cropped_raw_1) // 2
        num_elements_2 = len(cropped_raw_2) // 2
        ch1_data = struct.unpack('<' + ('h' * num_elements_1), cropped_raw_1[:num_elements_1*2])
        ch2_data = struct.unpack('<' + ('h' * num_elements_2), cropped_raw_2[:num_elements_2*2])
        min_length=np.min([len(ch1_data), len(ch2_data)])
        max_length=np.max([len(ch1_data), len(ch2_data)])
    
        #Recover voltage from raw data.
        ch1_volt = adc_to_volt(ch1_data, parse_volt_per_div(ch1_scale_str))[0:min_length]
        ch2_volt = adc_to_volt(ch2_data, parse_volt_per_div(ch2_scale_str))[0:min_length]
        
        #Recover time from raw data.
        sample_rate = min_length / (parse_time_per_div(time_scale_str) * (15.2)*(min_length/max_length)) #15.2 total divs in time axis.
        t = np.arange(min_length) / sample_rate #The time scaling is compensated with the unequal read values from CH1 and CH2 inputs.

        results.append([t, ch1_volt, ch2_volt])
        time.sleep(wait_time)
        
    idx, shortest_list = min(enumerate(results[i][0] for i in range(len(results))), key=lambda x: len(x[1]))

    truncated_list = [
    [inner[:len(shortest_list)] for inner in middle] 
    for middle in results
    ]
    
    np_results = np.array(truncated_list)
    final_data = np_results.reshape(-1, 3)

    np.savetxt(
        filename,
        final_data,
        delimiter="\t",
        header="time[s]\tCH1[V]\tCH2[V] (num. of cycles: "+str(num_of_cycles)+")",
        comments=""
     )
    
    return scope, truncated_list

#====================================================================================================
#====================================================================================================
#====================================================================================================

#1D Plotter function to show the read data in a twin-y axis figure.

def plotter_single(x, y1, y2):
    fig, ax1 = plt.subplots()

    #Set the colors.
    color1 = "tab:red"
    color2 = "tab:blue"

    #Left axis for y1.
    ax1.plot(x, y1, color=color1, label="SQUID signal")
    ax1.set_xlabel("t (s)", fontsize=12)
    ax1.set_ylabel("V_CH1 (V)", color=color1, fontsize=12)
    ax1.tick_params(axis="both", labelsize=10)
    ax1.tick_params(axis="y", labelcolor=color1)
    ax1.grid(True)

    #Right axis for y2.
    ax2 = ax1.twinx()
    ax2.plot(x, y2, color=color2, label="Reference")
    ax2.set_ylabel("V_CH2 (V)", color=color2, fontsize=12)
    ax2.tick_params(axis="y", labelcolor=color2, labelsize=10)

    #Set the legend.
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="best")

    plt.tight_layout()
    plt.show()

#====================================================================================================
#====================================================================================================
#====================================================================================================
    
#2D Plotter function to show a colormap of the measured data. 

def plotter_colormap(data_list, col_index=1, cmap='viridis'):
    
    data_array = np.array(data_list) #Converting the to a numpy array.
    
    slice_to_plot = data_array[:, col_index, :]# Slicing: row: first index, coloumn: third index. second index =1, which is channel 1 voltage
    
    plt.figure(figsize=(10, 6))
    img = plt.imshow(slice_to_plot, cmap=cmap, aspect='auto', origin='lower')
    
    plt.colorbar(img, label='3D color plot')
    plt.xlabel('t (s)')
    plt.ylabel('P (a.u.)')
    plt.title('3D color plot')

    plt.tight_layout()
    plt.show()

#====================================================================================================
#====================================================================================================
#====================================================================================================
    
#2D Plotter function to show a waterfalls plot of the data.

def plotter_waterfall(data_list, col_index=1, x_offset=0.05, y_offset=0.5):
    data_array = np.array(data_list)
    slice_to_plot = data_array[:, col_index, :]
    
    num_lines = slice_to_plot.shape[0]
    num_points = slice_to_plot.shape[1]

    #X axis.
    x = np.linspace(0, 10, num_points) 

    #Defining the colorscale.
    colors = ["red", "yellow", "green", "blue"]
    cmap = LinearSegmentedColormap.from_list("custom_wf", colors, N=num_lines)

    plt.figure()

    for i in range(num_lines):
        #Calculation of the offset for each curves.
        #i=num_lines-1 is blue, assuming a cooldown.
        current_x_offset = i * x_offset
        current_y_offset = i * y_offset
        
        #Data of the actual curve.
        display_x = x + current_x_offset
        display_y = slice_to_plot[i, :] + current_y_offset
        
        plt.plot(display_x, display_y, 
                 color=cmap(i / (num_lines - 1)), 
                 linewidth=1.5, 
                 alpha=0.8)

    plt.xlabel('t (s)')
    plt.ylabel('V_CH_1 (V) + offset')
    plt.title('V-I Cooldown')
    
    #Grid on.
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()

#====================================================================================================
#====================================================================================================
#====================================================================================================

#2D Plotter function to show a derivative, smoothed (Gauss filtered) plot of the measured data. 

from scipy.ndimage import gaussian_filter

def plotter_derivative_refined(data_list, col_index=1, cmap='RdBu_r', 
                               xlim=None, ylim=None, vlim=None, sigma=1):
    data_array = np.array(data_list)
    slice_to_plot = data_array[:, col_index, :]
    
    #Smooth to reduce noise.
    #Defining sigma for smoothing.
    if sigma > 0:
        slice_to_plot = gaussian_filter(slice_to_plot, sigma=sigma)
    
    #Derivative of the curve along the x axis.
    derivative_slice = np.gradient(slice_to_plot, axis=1)
    
    plt.figure(figsize=(10, 6))
    
    #Colormap limits.
    if vlim:
        vmin, vmax = vlim
    else:
        #Calculation of a sensible limit depending on the noise.
        limit = np.nanpercentile(np.abs(derivative_slice), 95)
        vmin, vmax = -limit, limit

    img = plt.imshow(derivative_slice, cmap=cmap, aspect='auto', 
                     origin='lower', vmin=vmin, vmax=vmax)
    
    plt.colorbar(img, label='dV/dI (a.u.)')
    if xlim: plt.xlim(xlim)
    if ylim: plt.ylim(ylim)
        
    plt.xlabel('t (s)')
    plt.ylabel('P (a.u.)')
    
    plt.title(f'Smoothed Shapiro (sigma={sigma})')
    plt.show()

# Open & check communication

In [3]:
rm = pyvisa.ResourceManager()
print(rm)

ip_address="TCPIP0::192.168.1.72::3000::SOCKET" #Check the IP address and the port (192.168.1.x::YYYY). Set static IP on the PC if it is not.

scope = rm.open_resource(ip_address) #Open communication.

scope.timeout = 500 #Set timeout in ms. Recommended is >5000 for readout.

scope.write_termination = '\r\n' #Set terminator commands.
scope.query_termination = '\r\n'
scope.read_termination = '\r\n'

print(scope) #Print the type of the scope.

time.sleep(0.1)

print(scope.query("*IDN?")) #Check communication with "*IDN? and print."

time.sleep(0.1)

scope.write("*CLS") #Clear the register.
#scope.write("*RST") #Restart. Uncomment if needed.
scope.clear() #Clear buffer and wait.
time.sleep(1.0)

Resource Manager of Visa Library at C:\WINDOWS\system32\visa32.dll
TCPIPSocket at TCPIP0::192.168.1.72::3000::SOCKET


C:\Users\Oliver\anaconda3\envs\owon\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()


OWON,XDS3102A,23300084,V8.2.0->



# ---START WORKING HERE---